In [1]:
import os
import json
from dotenv import load_dotenv
from binance.client import Client
from binance.exceptions import BinanceAPIException

# 1. Load environment variables from .env
load_dotenv()

def initialize_testnet_client():
    api_key = os.getenv("BINANCE_API_KEY")
    api_secret = os.getenv("BINANCE_API_SECRET")

    if not api_key or not api_secret:
        raise ValueError("API Keys not found. Check your .env file.")

    # 2. Initialize with testnet=True 
    # This automatically sets the correct URLs and signing logic
    return Client(api_key, api_secret, testnet=True)

client = initialize_testnet_client()

try:
    # Check Account Balance (Signed Request)
    # This is the best way to verify if your 401 error is fixed
    account = client.get_account()
    
    print("--- Account Status ---")
    # Filter for assets that actually have a balance in the testnet
    balances = [b for b in account['balances'] if float(b['free']) > 0]
    print(json.dumps(balances, indent=2))

    # Check Price (Unsigned Request)
    ticker = client.get_symbol_ticker(symbol="BTCUSDT")
    print(f"\n--- Market Price --- \n{ticker['symbol']}: {ticker['price']}")

    # Place a Test Order
    print("\n--- Sending Test Order ---")
    test_order = client.create_test_order(
        symbol='BTCUSDT',
        side='BUY',
        type='MARKET',
        quantity=0.01
    )
    print("✓ Connectivity check complete. Authenticated successfully.")

except BinanceAPIException as e:
    print(f"Binance Error: {e.status_code} - {e.message}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Account Status ---
[
  {
    "asset": "\u8fd9\u662f\u6d4b\u8bd5\u5e01",
    "free": "10000.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "456",
    "free": "10000.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "BNB",
    "free": "1.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "BTC",
    "free": "1.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "USDT",
    "free": "10000.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "ETH",
    "free": "1.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "LTC",
    "free": "8.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "TRX",
    "free": "1450.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "XRP",
    "free": "351.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "GAS",
    "free": "300.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "KNC",
    "free": "3374.00000000",
    "locked": "0.00000000"
  },
  {
    "asset": "IOTA"

In [2]:
try:
    print("\n--- Executing Actual Testnet Market Buy ---")
    # This replaces create_test_order with a real order on the testnet
    buy_order = client.order_market_buy(
        symbol='BTCUSDT',
        quantity=0.01
    )
    print(f"Order ID: {buy_order['orderId']} | Status: {buy_order['status']}")
    
    # Verify balance change
    new_balance = client.get_asset_balance(asset='BTC')
    print(f"New BTC Balance: {new_balance['free']}")

except BinanceAPIException as e:
    print(f"Trade Error: {e.status_code} - {e.message}")


--- Executing Actual Testnet Market Buy ---
Order ID: 1169159 | Status: FILLED
New BTC Balance: 1.00999000


In [14]:
# Use Binance Spot testnet for safety in this section
import os
import json
from binance.exceptions import BinanceAPIException

# Do not reference `client` before it is created. Use a constant for the testnet URL and
# set client.API_URL after initializing the Client in the initialization cell.
TESTNET_API_URL = 'https://testnet.binance.vision/api'

# Load API credentials from environment (recommended) with safe placeholders for examples.
API_KEY = os.getenv("BINANCE_API_KEY", "test_key")
API_SECRET = os.getenv("BINANCE_API_SECRET", "test_secret")

if API_KEY == "test_key" or API_SECRET == "test_secret":
    print("Warning: Using placeholder API keys. Set BINANCE_API_KEY and BINANCE_API_SECRET in your environment or .env for real credentials.")
    print("  Example (bash):")
    print("    export BINANCE_API_KEY=your_api_key")
    print("    export BINANCE_API_SECRET=your_api_secret")
    print("\\nFor now, we'll use testnet placeholders for examples.")
else:
    print("✓ API keys loaded successfully (not printing keys)")

# Example of creating a live order (COMMENTED OUT FOR SAFETY)
# Uncomment only when you're ready to execute real trades and you have initialized `client`.
def create_buy_order_example():
    """
    Example function to create a limit buy order.
    Requires `client` to be initialized (e.g., client = Client(API_KEY, API_SECRET))
    and, if desired, client.API_URL set to TESTNET_API_URL for testnet usage.
    """
    try:
        order = client.order_limit_buy(
            symbol='BTCUSDT',
            quantity=0.001,
            price=30000  # Set a reasonable price
        )
        print("=== Live Order Placed ===")
        print(json.dumps(order, indent=2))
        return order.get('orderId')
    except BinanceAPIException as e:
        status = getattr(e, 'status_code', None)
        msg = getattr(e, 'message', str(e))
        if status == -1013:
            print(f"Invalid quantity: {msg}")
        elif status == -2010:
            print(f"Insufficient balance: {msg}")
        else:
            print(f"Binance API Error: {status} - {msg}")
    except Exception as e:
        print(f"Error placing order: {e}")
    return None

# Safety message: function defined but not executed
print("Live order function is defined but not executed (for safety)")
print("Uncomment the create_buy_order_example() call to execute (ensure `client` is initialized and you understand the risks).")

✓ API keys loaded successfully (not printing keys)
Live order function is defined but not executed (for safety)
Uncomment the create_buy_order_example() call to execute (ensure `client` is initialized and you understand the risks).


## 3. Initialize REST Client

Create a Binance client instance. You can use either the main Binance API or the testnet for safe practice.

In [15]:
from binance.client import Client
from binance.exceptions import BinanceAPIException, BinanceRequestException

# Initialize client (using testnet by default for safety)
# For production, remove the tld='com' parameter
client = Client(API_KEY, API_SECRET)

# To use testnet instead, uncomment the line below:
# client = Client(API_KEY, API_SECRET, tld='com')
# To use the actual API (with real money), use the production URLs

print("✓ Binance client initialized successfully")

✓ Binance client initialized successfully


In [16]:
API_KEY, API_SECRET

('QCfdcTjQGyLbRJBAexBS2pa4ubhd07GhEacBLWArKZKpOX5m7arUCFNjhK5qh8KI',
 'x85r2e6jA2iYUCavHqiy8OtYddFCpfREY0zxLDmnVrP5Ue5WYaRNENDH8h2DWRov')

## 4. Test Connectivity

Verify that the connection to Binance is working by pinging the server and checking the server time.

In [17]:
import json
from datetime import datetime

try:
    # Test ping - should return empty dict if successful
    ping = client.ping()
    print(f"✓ Ping successful: {ping}")
    
    # Get server time
    server_time = client.get_server_time()
    server_time_ms = server_time.get('serverTime', 0)
    server_datetime = datetime.fromtimestamp(server_time_ms / 1000)
    
    print(f"✓ Server time: {server_datetime}")
    print(f"  (Unix timestamp: {server_time_ms})")
    
except BinanceAPIException as e:
    print(f"Binance API Error: {e.status_code} - {e.message}")
except BinanceRequestException as e:
    print(f"Connection Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

✓ Ping successful: {}
✓ Server time: 2026-05-08 21:25:48.969000
  (Unix timestamp: 1778246748969)


## 5. Fetch Market Data

Get real-time market data including ticker prices, order book depth, and candlestick (kline) data.

In [18]:
import pandas as pd

try:
    # 1. Get symbol ticker (current price)
    ticker = client.get_symbol_ticker(symbol='BTCUSDT')
    print("=== BTCUSDT Ticker ===")
    print(json.dumps(ticker, indent=2))
    print()
    
    # 2. Get order book
    order_book = client.get_order_book(symbol='BTCUSDT', limit=5)
    print("=== Order Book (Top 5) ===")
    print(f"Bids (Buy orders):")
    for bid in order_book['bids']:
        print(f"  Price: {bid[0]}, Quantity: {bid[1]}")
    print(f"\nAsks (Sell orders):")
    for ask in order_book['asks']:
        print(f"  Price: {ask[0]}, Quantity: {ask[1]}")
    print()
    
    # 3. Get candlestick data (klines)
    klines = client.get_klines(symbol='BTCUSDT', interval='1m', limit=5)
    print("=== BTCUSDT 1-minute Candlesticks (Last 5) ===")
    
    # Format klines data into a DataFrame for better visualization
    df = pd.DataFrame(klines, columns=['Open_time', 'Open', 'High', 'Low', 'Close', 
                                        'Volume', 'Close_time', 'Quote_asset_volume',
                                        'Trades', 'Taker_buy_base', 'Taker_buy_quote', 'Ignore'])
    df['Open_time'] = pd.to_datetime(df['Open_time'], unit='ms')
    df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
    
    print(df[['Open_time', 'Open', 'High', 'Low', 'Close', 'Volume']])
    
except BinanceAPIException as e:
    print(f"Binance API Error: {e.status_code} - {e.message}")
except Exception as e:
    print(f"Error fetching market data: {e}")

=== BTCUSDT Ticker ===
{
  "symbol": "BTCUSDT",
  "price": "79695.40000000"
}

=== Order Book (Top 5) ===
Bids (Buy orders):
  Price: 79695.40000000, Quantity: 6.55366000
  Price: 79695.39000000, Quantity: 0.43058000
  Price: 79695.05000000, Quantity: 0.00250000
  Price: 79695.04000000, Quantity: 0.00036000
  Price: 79695.03000000, Quantity: 0.05790000

Asks (Sell orders):
  Price: 79695.41000000, Quantity: 1.04266000
  Price: 79695.42000000, Quantity: 0.00064000
  Price: 79695.51000000, Quantity: 0.00022000
  Price: 79695.52000000, Quantity: 0.13815000
  Price: 79695.77000000, Quantity: 0.00014000

=== BTCUSDT 1-minute Candlesticks (Last 5) ===
            Open_time      Open      High       Low     Close   Volume
0 2026-05-08 13:21:00  79767.56  79769.84  79753.25  79753.26  5.58406
1 2026-05-08 13:22:00  79753.25  79753.26  79720.20  79749.27  7.33627
2 2026-05-08 13:23:00  79749.26  79757.46  79745.12  79745.12  5.10739
3 2026-05-08 13:24:00  79745.13  79747.53  79730.00  79737.74 

## 6. Fetch Account Info and Balances

Retrieve your account information and current asset balances. **Note:** This requires your API key to have appropriate permissions.

In [19]:
try:
    # Get account information
    account = client.get_account()
    
    print("=== Account Summary ===")
    print(f"Can Trade: {account['canTrade']}")
    print(f"Can Deposit: {account['canDeposit']}")
    print(f"Can Withdraw: {account['canWithdraw']}")
    print(f"Number of Assets: {len(account['balances'])}")
    print()
    
    # Display non-zero balances
    print("=== Asset Balances (Non-zero) ===")
    balances = []
    for balance in account['balances']:
        free = float(balance['free'])
        locked = float(balance['locked'])
        if free > 0 or locked > 0:
            balances.append({
                'Asset': balance['asset'],
                'Free': free,
                'Locked': locked,
                'Total': free + locked
            })
    
    if balances:
        df_balances = pd.DataFrame(balances)
        print(df_balances.to_string(index=False))
    else:
        print("No assets with balance found")
    
    # Alternative: Get specific asset balance
    print("\n=== Specific Asset: BTC ===")
    btc_balance = client.get_asset_balance('BTC')
    if btc_balance:
        print(json.dumps(btc_balance, indent=2))
    else:
        print("BTC balance: 0")
        
except BinanceAPIException as e:
    print(f"API Error: {e.status_code} - {e.message}")
    print("(This may require API key with proper permissions)")
except Exception as e:
    print(f"Error: {e}")

API Error: 401 - Invalid API-key, IP, or permissions for action.
(This may require API key with proper permissions)


## 7. Create Test Order

Use the test endpoint to validate order parameters without actually executing the order. This is **highly recommended** before placing real orders.

In [13]:
try:
    # Create a test order
    test_order = client.create_test_order(
        symbol='BTCUSDT',
        side='BUY',
        type='MARKET',
        quantity=0.001,
    )
    
    print("=== Test Order Result ===")
    print(json.dumps(test_order, indent=2))
    print("\n✓ Test order successful")
    
except Exception as e:
    print(f"Error: {e}")

Error: APIError(code=-2015): Invalid API-key, IP, or permissions for action.


## 8. Create Live Order (CAUTION!)

**⚠️ WARNING:** This will execute an actual order on your account. Only uncomment if you intend to trade real funds. Start with small quantities for testing.

In [ ]:
# Example of creating a live order (COMMENTED OUT FOR SAFETY)
# Uncomment only when you're ready to execute real trades

def create_buy_order_example():
    """
    Example function to create a limit buy order
    """
    try:
        order = client.order_limit_buy(
            symbol='BTCUSDT',
            quantity=0.001,
            price=30000  # Set a reasonable price
        )
        print("=== Live Order Placed ===")
        print(json.dumps(order, indent=2))
        return order['orderId']
        
    except BinanceAPIException as e:
        if e.status_code == -1013:
            print(f"Invalid quantity: {e.message}")
        elif e.status_code == -2010:
            print(f"Insufficient balance: {e.message}")
        else:
            print(f"Binance API Error: {e.status_code} - {e.message}")
    except Exception as e:
        print(f"Error placing order: {e}")
    
    return None

# To place an actual order, call:
# order_id = create_buy_order_example()

print("Live order function is defined but not executed (for safety)")
print("Uncomment the create_buy_order_example() call to execute")

## 9. WebSocket: Real-time Market Data

Subscribe to real-time market data streams using WebSocket for low-latency updates.

In [ ]:
# Use Binance testnet WebSocket endpoint for streaming examples
# We'll show a minimal websocket-client example pointed at testnet
TESTNET_WS = 'wss://testnet.binance.vision/ws'
import websocket, json, threading, time
def on_message(ws, message):
    msg = json.loads(message)
    print(msg)

def on_error(ws, error):
    print('WebSocket error:', error)

def on_close(ws, close_status_code, close_msg):
    print('WebSocket closed')

def on_open(ws):
    print('WebSocket opened (testnet)')
    # Subscribe to trade stream for BTCUSDT
    # example payload for aggregated trade stream subscription (if needed)
    # ws.send(json.dumps({"method":"SUBSCRIBE", "params": ["btcusdt@trade"], "id": 1}))

# Example: create but do not run the WebSocket client (uncomment to run)
# ws = websocket.WebSocketApp(TESTNET_WS + '/btcusdt@trade', on_message=on_message, on_error=on_error, on_close=on_close)
# wst = threading.Thread(target=ws.run_forever, kwargs={'ping_interval': 20})
# wst.start()

print("WebSocket testnet example defined. Use websocket-client to connect to TESTNET_WS.")
print("Note: Running websockets will block; run in separate thread or process.")

## 10. Error Handling and Retries

Implement robust error handling for network issues, rate limits, and API errors using the `tenacity` library for retries.

In [ ]:
# Ensure subsequent requests go to testnet when retrying/handling errors
client.API_URL = 'https://testnet.binance.vision'
from tenacity import retry, wait_exponential, stop_after_attempt
import requests

@retry(wait=wait_exponential(multiplier=1, min=1, max=10), stop=stop_after_attempt(3))
def fetch_market_data_with_retry(symbol='BTCUSDT'):
    """
    Fetch market data with automatic retry on failure
    """
    try:
        ticker = client.get_symbol_ticker(symbol=symbol)
        return ticker
    except BinanceAPIException as e:
        if e.status_code == -1003:  # Rate limit exceeded
            print(f"Rate limited. Retrying...")
            raise  # Let tenacity handle the retry
        else:
            print(f"Binance API Error: {e.message}")
            raise
    except BinanceRequestException as e:
        print(f"Connection error. Retrying...")
        raise  # Let tenacity handle the retry
    except requests.exceptions.Timeout:
        print(f"Request timeout. Retrying...")
        raise

# Test the retry decorator
print("Testing fetch_market_data_with_retry()...")
try:
    result = fetch_market_data_with_retry('BTCUSDT')
    print(f"✓ Successfully fetched: BTC = ${result['price']}")
except Exception as e:
    print(f"Failed after retries: {e}")

# Common error codes
error_codes = {
    -1000: "Invalid request payload",
    -1001: "Too many requests",
    -1003: "Rate limit exceeded",
    -1013: "Invalid quantity",
    -1015: "Too many orders",
    -2010: "Insufficient balance",
    -2015: "Invalid API key"
}

print("\n=== Common Binance Error Codes ===")
for code, description in error_codes.items():
    print(f"{code}: {description}")

## 11. Rate Limits, Logging and Headers

Monitor rate limit consumption and implement logging for debugging API interactions.

In [ ]:
# Ensure logging and rate-limit checks target testnet for examples
client.API_URL = 'https://testnet.binance.vision'
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for binance
binance_logger = logging.getLogger('binance')
binance_logger.setLevel(logging.DEBUG)

logger = logging.getLogger(__name__)

# Function to monitor rate limits
def fetch_with_rate_limit_monitoring():
    """
    Fetch data and monitor rate limit headers
    """
    try:
        # Make a request
        response = client.get_symbol_ticker(symbol='BTCUSDT')
        
        # Log the response
        logger.info(f"Successfully fetched ticker for BTCUSDT")
        
        # The python-binance library doesn't expose response headers directly,
        # but you can access them through a lower-level approach if needed
        print("✓ Request successful")
        print(f"  Price: ${response['price']}")
        
        # Best practice: Implement a simple backoff strategy
        logger.info("Waiting 100ms to respect rate limits...")
        time.sleep(0.1)
        
    except Exception as e:
        logger.error(f"Error fetching ticker: {e}")

# Rate limit info
print("=== Binance Rate Limits (examples using testnet) ===")
print("1400 requests per minute (REST API)")
print("10 orders per second")
print("100,000 requests per 24 hours (weight-based)")
print()

# Initialize and test logging
logger.info("Starting rate limit monitoring example")
fetch_with_rate_limit_monitoring()
logger.info("Completed rate limit monitoring example")

## 12. Basic pytest Unit Tests

Write unit tests with mocked responses to validate your integration without making real API calls.

In [ ]:
# Example test file content (save as test_binance_integration.py)

test_code = '''
import pytest
from unittest.mock import patch, MagicMock
from binance.client import Client
from binance.exceptions import BinanceAPIException

@pytest.fixture
def client_fixture():
    """Fixture to create a Binance client"""
    return Client("test_key", "test_secret")

def test_client_initialization(client_fixture):
    """Test that client initializes correctly"""
    assert client_fixture is not None
    assert client_fixture.API_KEY == "test_key"

@patch('binance.client.Client.ping')
def test_ping_success(mock_ping, client_fixture):
    """Test successful ping to Binance"""
    mock_ping.return_value = {}
    result = client_fixture.ping()
    assert result == {}
    mock_ping.assert_called_once()

@patch('binance.client.Client.get_symbol_ticker')
def test_get_ticker(mock_ticker, client_fixture):
    """Test fetching ticker data"""
    mock_ticker.return_value = {
        'symbol': 'BTCUSDT',
        'price': '45000.00'
    }
    result = client_fixture.get_symbol_ticker(symbol='BTCUSDT')
    assert result['symbol'] == 'BTCUSDT'
    assert result['price'] == '45000.00'

@patch('binance.client.Client.get_symbol_ticker')
def test_api_exception(mock_ticker, client_fixture):
    """Test handling of API exceptions"""
    mock_ticker.side_effect = BinanceAPIException(-2015, "Invalid API key")
    
    with pytest.raises(BinanceAPIException):
        client_fixture.get_symbol_ticker(symbol='BTCUSDT')

if __name__ == "__main__":
    print("Test file content generated above")
    print("To run tests, save as test_binance_integration.py and run: pytest test_binance_integration.py -v")
'''

print("=== Example Test File ===")
print(test_code)
print("\n✓ To use these tests, save the code above as 'test_binance_integration.py' in your project directory")

## 13. Run and Debug in VSCode

Execute code from the integrated terminal and view outputs. Create tasks for common operations.

### Common VSCode Commands

**Run Jupyter Notebook:**
- In Jupyter: Click "Run All" button
- Or press `Ctrl+Enter` (or `Cmd+Enter` on Mac) to run individual cells

**Run Python Scripts in Terminal:**
```bash
python your_script.py
```

**Run Tests:**
```bash
# Run all tests
pytest

# Run with verbose output
pytest test_binance_integration.py -v

# Run specific test
pytest test_binance_integration.py::test_ping_success -v

# Run with coverage
pytest --cov=.
```

**Debug Python Script:**
1. Set breakpoints by clicking in the left margin
2. Press `F5` or Run > Start Debugging
3. Use Debug Console to inspect variables

**Useful VSCode Extensions:**
- Python (Microsoft)
- Pylance - Fast Python language server
- Python Test Explorer - For running tests from UI

## Summary & Next Steps

### What You've Learned:
✓ Installing and configuring the Binance Python client  
✓ Authenticating with API credentials securely  
✓ Fetching real-time market data (tickers, order books, candles)  
✓ Accessing account information and balances  
✓ Placing test and live orders  
✓ Subscribing to real-time WebSocket data  
✓ Implementing error handling and retry logic  
✓ Monitoring rate limits  
✓ Writing unit tests with mocked responses  

### Next Steps:
1. **Create a `.env` file** with your Binance API credentials
2. **Test with testnet** before using real funds
3. **Implement a trading strategy** using the market data
4. **Add monitoring and logging** to track API usage
5. **Set up alerts** for rate limits and errors
6. **Read Binance API Documentation**: https://binance-docs.github.io/apidocs/

### Useful Resources:
- [python-binance Documentation](https://python-binance.readthedocs.io/)
- [Binance API Reference](https://binance-docs.github.io/apidocs/)
- [Binance Testnet](https://testnet.binance.vision/)

### Security Best Practices:
- Never commit API keys to version control
- Use read-only keys for market data queries
- Use IP whitelisting on Binance
- Rotate API keys regularly
- Use testnet/sandbox for development and testing